# 0.9B 體育賽事預測模型 — Colab 訓練

在 Google Colab 的 GPU 上微調 sports prediction LLM(本 repo 的完整 pipeline)。

- **免費 Colab (T4 16GB)**:可跑 SFT + DPO + 評估;GRPO 請關掉(`USE_GRPO=0`)或換更高級 GPU
- **Colab Pro / A100**:全三階段
- 有真實資料:把 csv 上傳到 Colab 後修改下方 `MATCHES` 路徑(欄位見 repo 的 `data/SCHEMA.md`)

In [ ]:
# 0) 基本設定:把你的 repo clone URL 放進來(本 repo)
REPO_URL = "https://github.com/leo081127-hue/test-0.9b.git"
BRANCH   = "arena/01a0bd2e-test-0-9b"  # 或 "main"(合併後)
# 基座模型:K2-Horizon 或 Qwen 備用
MODEL_ID   = "IFM/K2-Horizon-0.9B"
EXTRA_ARGS = "--trust-remote-code"   # K2-Horizon 需要;換 Qwen 時改成 ""
USE_GRPO   = "0"                     # T4 建議 0;A100 可開 1
!git clone -b {BRANCH} {REPO_URL} repo 2>/dev/null || echo "(already cloned)"
!cd repo && pip install -q -r requirements.txt {('trl datasets' if USE_GRPO == '1' else '')}
%cd /content/repo

## 1) 資料
預設產生合成 demo 資料(1800 場)。要換自己的:先上傳 `你的資料.csv` 到 Colab,再把 `MATCHES` 指過去。

In [ ]:
MATCHES = "data/demo/matches.csv"  # 改成你上傳的 csv 路徑
!python data/generate_demo_data.py --out $MATCHES
!python data/qa_report.py --matches $MATCHES   # 資料品質檢查

## 2) 訓練(SFT → DPO → 評估,GRPO 可選)
T4 上 SFT 約 30~90 分鐘(1800 場 × 3 epochs)。輸出在 `output/`。

In [ ]:
!python data/build_dataset.py --matches $MATCHES --out data/out
!python train/sft.py $EXTRA_ARGS --model-id $MODEL_ID \
    --train-jsonl data/out/train.jsonl --val-jsonl data/out/val.jsonl \
    --adapter-dir output/sft --epochs 3 --batch-size 4 --grad-accum 4 --grad-ckpt
!python train/dpo.py $EXTRA_ARGS --model-id $MODEL_ID \
    --adapter-dir output/sft --pairs data/out/dpo_pairs.jsonl \
    --output-dir output/dpo --epochs 1 --lr 5e-5
%env USE_GRPO=$USE_GRPO
!python eval/evaluate.py $EXTRA_ARGS --model-id $MODEL_ID --adapter-dir output/dpo \
    --test-jsonl data/out/test.jsonl --matches-csv $MATCHES \
    --report output/report.json --plot output/calibration.png --pred-out output/preds.csv

In [ ]:
# 可選:GRPO(RL 階段,需 trl;A100 建議)
if USE_GRPO == "1":
    get_ipython().system(
        f"python train/grpo.py {EXTRA_ARGS} --model-id {MODEL_ID} "
        "--train-jsonl data/out/train.jsonl --adapter-dir output/dpo "
        "--output-dir output/grpo --num-generations 8 --batch-size 8 --max-completion 512"
    )
    get_ipython().system(
        f"python eval/evaluate.py {EXTRA_ARGS} --model-id {MODEL_ID} --adapter-dir output/grpo "
        f"--test-jsonl data/out/test.jsonl --matches-csv {MATCHES} "
        "--report output/report_grpo.json --plot output/calibration_grpo.png"
    )

## 3) 回測:贏錢了嗎?
CLV > 0 且 ROI > 0(相對 market_open 基準)才算有意義。

In [ ]:
!python eval/backtest.py --preds output/preds.csv --matches-csv $MATCHES \
    --threshold 0.55 --report output/backtest.json --plot output/equity.png

## 4) 下載產物
adapter 很小(LoRA),可以直接下載回本機/自己的 GPU 機器部署。

In [ ]:
from google.colab import files
!zip -qr output/adapter_dpo.zip output/dpo
files.download("output/adapter_dpo.zip")
files.download("output/report.json")
files.download("output/equity.png")